# 04 - Graph Construction

**Architecture block 2c.** Combine location + time -> KNN + radius search ->
spatial graph.

A node is one observation. Edges join observations close in space *and* time,
plus directed downwind edges for wind-borne spore transport.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))

import cropforecast
from cropforecast.config import load_config, ensure_dirs, set_seed, Device
cfg = load_config(Path.cwd().parent / "configs" / "default.yaml")
ensure_dirs(cfg); set_seed(cfg.project.seed)
device = Device.auto(cfg.training.amp)
print("device:", device)

In [ ]:
from cropforecast.graph.build import haversine_km, initial_bearing_deg, site_distance_matrix
from cropforecast.data.farms import to_frame
sites = to_frame()
d, b, ids = site_distance_matrix(sites)
print(f"Nashik -> Sangli : {haversine_km(19.9975,73.7898,16.8524,74.5815):.1f} km (real ~360)")
print(f"bearing          : {initial_bearing_deg(19.9975,73.7898,16.8524,74.5815):.0f} deg")

### Calibrating the radius

Indian production belts are far apart, so a small radius disconnects the graph.

In [ ]:
import numpy as np
rows = []
for r in [150, 250, 350, 500]:
    A = (d < r) & (d > 0)
    rows.append({"radius_km": r, "mean_degree": A.sum(1).mean().round(2),
                 "isolated_sites": int((A.sum(1) == 0).sum())})
import pandas as pd; pd.DataFrame(rows)

At 150 km a third of the sites are orphaned. We use 350 km **plus** a KNN
fallback (`min_neighbours`) so no farm is ever isolated - the diagram does say
"KNN *and* radius search".

In [ ]:
from cropforecast.graph.build import build_graph
obs = pd.read_parquet(Path(cfg.paths.processed) / "observations.parquet")
sample = obs.sample(4000, random_state=cfg.project.seed).reset_index(drop=True)
g = build_graph(sample, sites,
                k_neighbours=cfg.graph.k_neighbours, radius_km=cfg.graph.radius_km,
                time_window_days=cfg.graph.time_window_days,
                max_edges_per_node=cfg.graph.max_edges_per_node,
                wind_aware=True, min_neighbours=cfg.graph.min_neighbours)
g.summary()

### Downwind edges

Directed edges added where the wind at the source actually blew toward the target.

In [ ]:
print("edge attributes:", g.ATTR_NAMES)
print(f"downwind edges: {(g.edge_type==1).sum():,} of {g.edge_index.shape[1]:,}")
pd.DataFrame(g.edge_attr[:8], columns=list(g.ATTR_NAMES)).round(3)

## Wind-borne spread between farms

The graph only has something to learn if a farm's future depends on its
*neighbours*, not just on itself. This is the epidemic that supplies that
dependence.

In [ ]:
from cropforecast.physics.contagion import simulate_crop, spatial_autocorrelation, EpidemicParams
climate = pd.read_parquet(Path(cfg.paths.processed) / "climate_features.parquet")
climate["date"] = pd.to_datetime(climate["date"])
print("epidemic constants:", EpidemicParams())
sim = simulate_crop(climate, "Tomato")
print(f"{len(sim):,} crop-site-days simulated")
sim.infection_pressure.describe().round(4)

### The decisive measurement

Compare two predictors of a farm's infection pressure *h* days ahead: its own
current pressure, and its neighbours' current pressure.

In [ ]:
rows = [spatial_autocorrelation(sim, climate, "Tomato", lag_days=h) for h in (1, 3, 7)]
pd.DataFrame(rows)

At longer lags the neighbours are the *better* predictor. That is the signal a
GNN can exploit and a per-node model cannot.

In [ ]:
# An outbreak travelling across the network
wide = sim.pivot(index="date", columns="site_id", values="infection_pressure")
peak = np.unravel_index(np.nanargmax(wide.to_numpy()), wide.shape)
window = wide.iloc[max(0, peak[0]-25):peak[0]+10]
fig, ax = plt.subplots(figsize=(13, 5))
for sid in window.columns:
    ax.plot(window.index, window[sid], lw=1.6, label=sid)
ax.set_ylabel("infection pressure"); ax.set_title("An epidemic moving between farms")
ax.legend(ncol=4, fontsize=8); ax.grid(alpha=.3); plt.show()

### The farm network

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 10))
k, radius, minnb = cfg.graph.k_neighbours, cfg.graph.radius_km, cfg.graph.min_neighbours
for i, sid in enumerate(ids):
    order = [j for j in d[i].argsort() if j != i]
    within = [j for j in order if d[i, j] <= radius][:k] or []
    if len(within) < minnb: within = order[:minnb]
    for j in within:
        ax.plot([sites.lon[i], sites.lon[j]], [sites.lat[i], sites.lat[j]],
                color="#22c55e", alpha=.25, lw=.8, zorder=1)
ax.scatter(sites.lon, sites.lat, s=70, c="#f43f5e", zorder=2, edgecolor="white")
for _, r in sites.iterrows():
    ax.annotate(r["name"], (r.lon, r.lat), fontsize=7, xytext=(3,3), textcoords="offset points")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Spatial graph over 37 Indian farm districts"); ax.grid(alpha=.2); plt.show()